# LeetCode 664 — Strange Printer bằng Python

- Link: https://leetcode.com/problems/strange-printer/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên đoạn (Interval DP), Đệ quy có nhớ, BFS trên trạng thái

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def strangePrinter(self, s: str) -> int:

```

## 1. Giới thiệu & mục tiêu học

Strange Printer là bài **DP trên đoạn** (interval DP) kinh điển, cùng họ với Remove Boxes, Burst Balloons, Zuma. Đặc trưng: trạng thái là một đoạn `[i, j]`, và ta "quyết định số phận" của một phần tử đặc biệt (ở đây là ký tự đầu đoạn) bằng cách xem nó **đi chung lượt in** với những ký tự nào.

Sau notebook này bạn sẽ:

1. Hiểu đúng luật của máy in lạ và vì sao đếm "số đoạn ký tự liên tiếp" là **sai**.
2. Viết brute force bằng **BFS trên trạng thái chuỗi** để có đáp án chuẩn cho chuỗi ngắn.
3. Xây dựng công thức DP `f(i, j)`: ký tự `s[i]` in riêng, hoặc in chung một lượt với một ký tự `s[k] == s[i]` phía sau.
4. Cài đặt bằng đệ quy có nhớ, bottom-up có nén chuỗi, và một công thức chia đôi khác — rồi đối chiếu tất cả với BFS.
5. Đọc được bảng DP theo độ dài đoạn và tự trace một ví dụ.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có một máy in lạ với hai tính chất:

1. Mỗi **lượt**, máy chỉ in **một dãy liên tiếp gồm cùng một ký tự**.
2. Mỗi lượt có thể bắt đầu và kết thúc ở **bất kỳ vị trí nào**, và ký tự mới **đè lên** ký tự cũ.

Cho chuỗi `s`, tìm **số lượt in ít nhất** để in ra đúng `s` (bắt đầu từ giấy trắng).

**Ví dụ 1**

```text
Input:  s = "aaabbb"
Output: 2
Giải thích: in "aaa" trước, rồi in "bbb".
```

**Ví dụ 2**

```text
Input:  s = "aba"
Output: 2
Giải thích: in "aaa" trước, rồi in "b" đè lên vị trí giữa.
```

**Ràng buộc**

- `1 <= s.length <= 100`
- `s` chỉ gồm chữ cái tiếng Anh thường.

`n <= 100` gợi ý ngay: lời giải `O(n^3)` (khoảng `10^6` phép) là hoàn toàn ổn.

In [1]:
def apply_prints(n, prints):
    """Mô phỏng máy in: prints là list (l, r, ch), in ch lên [l, r] (bao gồm hai đầu)."""
    paper = ['_'] * n
    for l, r, ch in prints:
        for p in range(l, r + 1):
            paper[p] = ch
        print(f"  in '{ch}' lên [{l},{r}] -> {''.join(paper)}")
    return ''.join(paper)

print("Ví dụ 2:")
assert apply_prints(3, [(0, 2, 'a'), (1, 1, 'b')]) == "aba"
print("Ví dụ khó hơn 'abcba' với 3 lượt:")
assert apply_prints(5, [(0, 4, 'a'), (1, 3, 'b'), (2, 2, 'c')]) == "abcba"

Ví dụ 2:
  in 'a' lên [0,2] -> aaa
  in 'b' lên [1,1] -> aba
Ví dụ khó hơn 'abcba' với 3 lượt:
  in 'a' lên [0,4] -> aaaaa
  in 'b' lên [1,3] -> abbba
  in 'c' lên [2,2] -> abcba


Ví dụ `"abcba"` cho thấy sức mạnh của việc **in đè**: 5 ký tự, 5 "đoạn" khác nhau, nhưng chỉ cần 3 lượt nhờ in lớp ngoài trước, lớp trong sau.

## 3. Kiến thức nền

### 3.1. Đệ quy có nhớ với `lru_cache`

Hàm đệ quy trên hai tham số `(i, j)` có `O(n^2)` trạng thái; `@lru_cache` đảm bảo mỗi trạng thái chỉ tính một lần.

In [2]:
from functools import lru_cache

@lru_cache(maxsize=None)
def paths(i, j):
    """Số đường đi từ (0,0) tới (i,j) chỉ đi xuống / sang phải."""
    if i == 0 or j == 0:
        return 1
    return paths(i - 1, j) + paths(i, j - 1)

print(paths(10, 10), paths.cache_info())

184756 CacheInfo(hits=81, misses=120, maxsize=None, currsize=120)


### 3.2. Bảng DP hai chiều và thứ tự điền theo độ dài đoạn

Với DP trên đoạn, `dp[i][j]` phụ thuộc các đoạn **ngắn hơn** nằm bên trong. Vì vậy ta điền theo **độ dài tăng dần**: mọi đoạn dài 1, rồi dài 2, ...

In [3]:
n = 4
order = []
for length in range(1, n + 1):
    for i in range(0, n - length + 1):
        j = i + length - 1
        order.append((i, j))
print(order)

[(0, 0), (1, 1), (2, 2), (3, 3), (0, 1), (1, 2), (2, 3), (0, 2), (1, 3), (0, 3)]


### 3.3. `itertools.groupby` — nén các ký tự liên tiếp giống nhau

`"aaabbb"` và `"ab"` cần cùng số lượt (in một ký tự hay nhiều ký tự liền nhau cùng loại đều tốn một lượt như nhau). Nén chuỗi giúp DP nhỏ hơn.

In [4]:
from itertools import groupby
for s in ["aaabbb", "aabbaa", "abc", "zzzz"]:
    print(s, "->", ''.join(ch for ch, _ in groupby(s)))

aaabbb -> ab
aabbaa -> aba
abc -> abc
zzzz -> z


### 3.4. BFS với `collections.deque`

BFS (tìm kiếm theo chiều rộng) duyệt trạng thái theo **số bước tăng dần**, nên lần đầu gặp trạng thái đích chính là số bước ít nhất. `deque` cho `popleft()` `O(1)`.

In [5]:
from collections import deque

def bfs_min_steps(start, target, moves):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        if u == target:
            return dist[u]
        for v in moves(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return -1

# Ví dụ nhỏ: từ 1 tới 10, mỗi bước +1 hoặc *2
print(bfs_min_steps(1, 10, lambda x: [x + 1, x * 2] if x < 20 else []))

4


### 3.5. `min` với vòng lặp và giá trị khởi tạo

Mẫu quen thuộc của DP: khởi tạo `res` bằng một lựa chọn chắc chắn hợp lệ, rồi duyệt các lựa chọn khác và lấy `min`.

In [6]:
options = [7, 3, 9, 4]
res = options[0]
for x in options[1:]:
    res = min(res, x)
print(res)

3


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đếm số đoạn có đúng không?

Một suy nghĩ đầu tiên: "mỗi đoạn ký tự liên tiếp giống nhau cần một lượt" → đáp án = số đoạn. Với `"aaabbb"` cho 2 ✓, nhưng `"aba"` cho 3 ✗ (đáp án 2). Lý do: lượt in `a` có thể **phủ cả hai đầu** và để `b` đè lên giữa.

### 4.2. Nhìn vào ký tự đầu tiên

Xét đoạn `s[i..j]`. Ký tự `s[i]` phải được in bởi **một lượt nào đó** (gọi là lượt `T`). Có hai khả năng:

- **`T` chỉ "lộ" ra ở vị trí `i`** — coi như in `s[i]` riêng: tốn 1 lượt + in phần còn lại `s[i+1..j]`.
- **`T` còn lộ ra ở vị trí khác**: gọi `k` là vị trí **gần nhất** sau `i` mà `T` lộ ra (nên `s[k] == s[i]`). Khi đó:
  - Đoạn giữa `s[i+1..k-1]` phải được in **hoàn toàn bằng các lượt khác**, và các lượt đó không thể vượt ra ngoài (nếu vượt qua `k` thì đã đè mất ký tự của `T` ở `k`).
  - Phần `s[k..j]` cùng với `T` (cắt từ `k` trở đi) là một cách in hợp lệ của `s[k..j]` — và lượt `T` đã được "tính" trong đó.

Công thức:

```text
f(i, j) = min( 1 + f(i+1, j),
               min over k in (i, j] với s[k] == s[i]:  f(i+1, k-1) + f(k, j) )
f(i, j) = 0 nếu i > j
```

### 4.3. Tính tay `s = "aba"`

- `f(2,2) = 1` ("a"), `f(1,1) = 1` ("b"), `f(1,2)` ("ba") `= 1 + f(2,2) = 2`.
- `f(0,2)`:
  - in `a` riêng: `1 + f(1,2) = 3`.
  - ghép với `k = 2` (`s[2] = 'a'`): `f(1,1) + f(2,2) = 1 + 1 = 2`.
  - → `f(0,2) = 2` ✓.

Ý nghĩa của nhánh thứ hai: lượt in `a` kéo dài từ 0 tới 2, chi phí của nó được tính trong `f(2,2)`; phần giữa `"b"` tốn thêm `f(1,1) = 1`.

### 4.4. Tính tay `s = "abcba"`

- `f(0,4)`: `s[0] = 'a'` có thể ghép với `k = 4`: `f(1,3) + f(4,4) = f("bcb") + 1`.
- `f(1,3)` ("bcb"): ghép `b` ở 1 với `b` ở 3: `f(2,2) + f(3,3) = 1 + 1 = 2`.
- → `f(0,4) = 2 + 1 = 3` ✓ (khớp cách in ở mục 2).

In [7]:
@lru_cache(maxsize=None)
def f_demo(s, i, j, depth=0):
    if i > j:
        return 0
    best = 1 + f_demo(s, i + 1, j, depth + 1)
    choice = "in riêng"
    for k in range(i + 1, j + 1):
        if s[k] == s[i]:
            cand = f_demo(s, i + 1, k - 1, depth + 1) + f_demo(s, k, j, depth + 1)
            if cand < best:
                best, choice = cand, f"ghép với k={k}"
    return best

for s in ["aba", "abcba", "aaabbb", "abab"]:
    print(s, "->", f_demo(s, 0, len(s) - 1))

aba -> 2
abcba -> 3
aaabbb -> 2
abab -> 3


## 5. Cách 1 — Brute Force: BFS trên trạng thái giấy

**Ý tưởng.** Trạng thái = chuỗi hiện có trên giấy (dùng `_` cho ô trắng). Một bước = chọn đoạn `[l, r]` và ký tự `c` (chỉ cần xét các ký tự có trong `s`), in đè. BFS từ giấy trắng tới `s`; số bước khi tới đích là đáp án.

Không có giả định thông minh nào → đây là "trọng tài" đáng tin cho chuỗi ngắn.

In [8]:
def strange_printer_bfs(s):
    n = len(s)
    letters = sorted(set(s))
    start = '_' * n

    def moves(state):
        for l in range(n):
            for r in range(l, n):
                for c in letters:
                    yield state[:l] + c * (r - l + 1) + state[r + 1:]

    return bfs_min_steps(start, s, moves)

print(strange_printer_bfs("aaabbb"), strange_printer_bfs("aba"), strange_printer_bfs("abcba"), strange_printer_bfs("abab"))
assert strange_printer_bfs("aaabbb") == 2
assert strange_printer_bfs("aba") == 2

2 2 3 3


**Độ phức tạp.** Số trạng thái tới `(m+1)^n` (`m` = số loại ký tự), mỗi trạng thái có `O(n^2 · m)` bước đi → **hàm mũ**. Với `n = 6, m = 3`: `4^6 = 4096` trạng thái × 63 bước — được; với `n = 100` thì không thể. Space cũng hàm mũ.

**Nút thắt.** BFS không tận dụng cấu trúc: các lượt in "lồng nhau" theo đoạn. Bài toán con tự nhiên là **đoạn** `s[i..j]`, chỉ có `O(n^2)` đoạn.

In [9]:
import time, random
random.seed(0)
for n in (4, 5, 6):
    s = ''.join(random.choice("abc") for _ in range(n))
    t = time.perf_counter()
    r = strange_printer_bfs(s)
    print(f"n={n}, s={s}: {r}  {time.perf_counter() - t:.3f}s")

n=4, s=bbab: 2  0.000s
n=5, s=cbbbb: 2  0.000s
n=6, s=bcacab: 4  0.060s


## 6. Bước đột phá

**Insight 1 (bài toán con là đoạn).** Nếu một lượt in lộ ra ở hai vị trí `i < k` mà không lộ ở giữa, thì mọi lượt in **sau nó** chạm vào khoảng `(i, k)` phải nằm **gọn trong** `(i, k)` — vì nếu chạm `i` hoặc `k` thì đã đè mất. Do đó đoạn giữa là một bài toán **độc lập**.

**Insight 2 (quyết định theo ký tự đầu).** Chỉ cần hỏi: lượt in tạo ra `s[i]` lộ ra lần kế tiếp ở đâu? Hoặc không ở đâu cả (in riêng), hoặc ở một `k` có `s[k] == s[i]`.

**Insight 3 (ký tự lặp liên tiếp miễn phí).** `s[i] == s[i+1]` thì `f(i, j) = f(i+1, j)` — ghép với `k = i+1` cho `f(i+1, i) + f(i+1, j) = 0 + f(i+1, j)`. Vì vậy nén chuỗi không làm thay đổi đáp án.

**Invariant DP:** `f(i, j)` = số lượt ít nhất để in đúng `s[i..j]` trên giấy trắng (đoạn độc lập).

**Quy tắc nếu…thì…:**
- Nếu `i > j` thì `0` lượt.
- Nếu `i == j` thì `1` lượt.
- Nếu có `k` với `s[k] == s[i]` thì thử "kéo dài lượt in của `s[i]` tới `k`": chi phí `f(i+1, k-1) + f(k, j)`.
- Luôn có lựa chọn in riêng: `1 + f(i+1, j)`.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | BFS trên trạng thái (mục 5) | hàm mũ | hàm mũ |
| 2 | Đệ quy có nhớ `f(i, j)` | `O(n^3)` | `O(n^2)` |
| 3 | Bottom-up + nén chuỗi | `O(m^3)`, `m <= n` | `O(m^2)` |
| 4 | Công thức chia đôi `dp[i][k] + dp[k+1][j]` | `O(n^3)` | `O(n^2)` |

### 7.1. Cách 2 — Đệ quy có nhớ

**Giải thích.** Cài đặt trực tiếp công thức mục 4.2.

In [10]:
def strange_printer_memo(s):
    n = len(s)

    @lru_cache(maxsize=None)
    def f(i, j):
        if i > j:
            return 0
        res = 1 + f(i + 1, j)                     # in s[i] riêng
        for k in range(i + 1, j + 1):
            if s[k] == s[i]:                      # kéo dài lượt in của s[i] tới k
                res = min(res, f(i + 1, k - 1) + f(k, j))
        return res

    return f(0, n - 1)

print(strange_printer_memo("aaabbb"), strange_printer_memo("aba"), strange_printer_memo("abcba"), strange_printer_memo("tbgtgb"))
assert strange_printer_memo("aaabbb") == 2 and strange_printer_memo("aba") == 2

2 2 3 4


**Đọc code:**

- `f(i + 1, k - 1)`: có thể là đoạn rỗng (khi `k = i + 1`) → trả 0.
- `f(k, j)`: đoạn bắt đầu bằng `s[k] == s[i]` — lượt in của `s[i]` được "gộp" vào lượt in `s[k]` trong bài toán con này, nên **không** cộng thêm 1.
- Độ sâu đệ quy tối đa khoảng `n = 100` — an toàn.

**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionMemo:
    def strangePrinter(self, s: str) -> int:
        s = ''.join(ch for ch, _ in groupby(s))      # nén để nhanh hơn
        n = len(s)

        @lru_cache(maxsize=None)
        def f(i, j):
            if i > j:
                return 0
            res = 1 + f(i + 1, j)
            for k in range(i + 1, j + 1):
                if s[k] == s[i]:
                    cand = f(i + 1, k - 1) + f(k, j)
                    if cand < res:
                        res = cand
            return res

        return f(0, n - 1)

assert SolutionMemo().strangePrinter("aba") == 2
print("SolutionMemo OK")

SolutionMemo OK


**Độ phức tạp Cách 2:** `O(n^2)` trạng thái × `O(n)` vòng `k` → Time **`O(n^3)`**, Space **`O(n^2)`**.

### 7.2. Cách 3 — Bottom-up có nén chuỗi

**Giải thích.** Nén chuỗi (Insight 3) để `m` = số đoạn. Điền `dp[i][j]` theo độ dài tăng dần, hoặc gọn hơn: `i` từ `m-1` về `0`, `j` từ `i` lên `m-1` (khi tính `dp[i][j]`, các ô `dp[i+1][*]` và `dp[k][j]` với `k > i` đã có).

In [12]:
def strange_printer_bottom_up(s):
    s = ''.join(ch for ch, _ in groupby(s))
    m = len(s)
    dp = [[0] * (m + 1) for _ in range(m + 1)]     # dp[i][j], quy ước dp[i][i-1] = 0
    for i in range(m - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, m):
            best = 1 + dp[i + 1][j]
            for k in range(i + 1, j + 1):
                if s[k] == s[i]:
                    inner = dp[i + 1][k - 1] if k - 1 >= i + 1 else 0
                    best = min(best, inner + dp[k][j])
            dp[i][j] = best
    return dp[0][m - 1]

print(strange_printer_bottom_up("aaabbb"), strange_printer_bottom_up("aba"), strange_printer_bottom_up("abcba"))
assert strange_printer_bottom_up("abcba") == 3

2 2 3


**Đọc code:**

- `i` chạy **ngược** vì `dp[i][j]` cần `dp[i+1][...]` và `dp[k][j]` (`k > i`).
- `inner = dp[i+1][k-1] if k-1 >= i+1 else 0`: đoạn rỗng khi `k = i+1`. (Sau khi nén, `s[i+1] != s[i]` nên `k = i+1` không bao giờ thỏa — nhưng giữ để code tổng quát.)
- Mảng kích thước `(m+1) × (m+1)` để `dp[i+1][j]` với `i = m-1` không vượt chỉ số.

**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionBottomUp:
    def strangePrinter(self, s: str) -> int:
        s = ''.join(ch for ch, _ in groupby(s))
        m = len(s)
        dp = [[0] * (m + 1) for _ in range(m + 1)]
        for i in range(m - 1, -1, -1):
            dp[i][i] = 1
            row_next = dp[i + 1]
            for j in range(i + 1, m):
                best = 1 + row_next[j]
                for k in range(i + 2, j + 1):            # k = i+1 không thỏa sau khi nén
                    if s[k] == s[i]:
                        cand = row_next[k - 1] + dp[k][j]
                        if cand < best:
                            best = cand
                dp[i][j] = best
        return dp[0][m - 1]

assert SolutionBottomUp().strangePrinter("aaabbb") == 2
print("SolutionBottomUp OK")

SolutionBottomUp OK


**Độ phức tạp Cách 3:** Time `O(m^3)`, Space `O(m^2)` với `m <= n` là độ dài sau nén.

### 7.3. Cách 4 — Công thức chia đôi

**Giải thích.** Một công thức khác cũng phổ biến (giống lời giải chính thức):

```text
dp[i][i] = 1
nếu s[i] == s[j]:  dp[i][j] = dp[i][j-1]          (ký tự cuối "đi nhờ" lượt in của s[i])
ngược lại:          dp[i][j] = min over k in [i, j-1] của dp[i][k] + dp[k+1][j]
```

Trực giác: nếu hai đầu cùng ký tự, lượt in `s[i]` có thể kéo dài tới `j` mà không tốn thêm. Nếu khác, **không có lượt nào** vừa lộ ở `i` vừa lộ ở `j`, nên tồn tại một điểm cắt `k` mà không lượt "lộ" nào bắc qua — bài toán tách làm hai.

In [14]:
def strange_printer_split(s):
    n = len(s)
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            if s[i] == s[j]:
                dp[i][j] = dp[i][j - 1]
            else:
                dp[i][j] = min(dp[i][k] + dp[k + 1][j] for k in range(i, j))
    return dp[0][n - 1]

print(strange_printer_split("aaabbb"), strange_printer_split("aba"), strange_printer_split("abcba"), strange_printer_split("tbgtgb"))
assert strange_printer_split("aba") == 2

2 2 3 4


**Đọc code:**

- Khi `s[i] == s[j]`: lấy kết quả của `s[i..j-1]`, ký tự `j` được phủ bằng cách kéo dài lượt in chứa `s[i]` (lượt đó trong lời giải của `s[i..j-1]` có thể giả sử là lượt đầu tiên phủ từ `i`).
- Khi khác nhau: thử mọi điểm cắt.
- So với Cách 2, công thức này thử nhiều điểm cắt hơn (không lọc theo `s[k] == s[i]`) nhưng cùng `O(n^3)`.

**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionSplit:
    def strangePrinter(self, s: str) -> int:
        s = ''.join(ch for ch, _ in groupby(s))
        n = len(s)
        dp = [[0] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            dp[i][i] = 1
            for j in range(i + 1, n):
                if s[i] == s[j]:
                    dp[i][j] = dp[i][j - 1]
                else:
                    best = n
                    for k in range(i, j):
                        v = dp[i][k] + dp[k + 1][j]
                        if v < best:
                            best = v
                    dp[i][j] = best
        return dp[0][n - 1]

assert SolutionSplit().strangePrinter("abcba") == 3
print("SolutionSplit OK")

SolutionSplit OK


**Độ phức tạp Cách 4:** Time `O(n^3)`, Space `O(n^2)`.

## 8. TRACE từng bước

Trace Cách 3 với `s = "abcabc"` (không cần nén). Bảng `dp[i][j]` (hàng `i`, cột `j`):

| i \ j | 0 a | 1 b | 2 c | 3 a | 4 b | 5 c |
|---|---|---|---|---|---|---|
| 0 | 1 | 2 | 3 | 3 | 4 | 5 |
| 1 |   | 1 | 2 | 3 | 3 | 4 |
| 2 |   |   | 1 | 2 | 3 | 3 |
| 3 |   |   |   | 1 | 2 | 3 |
| 4 |   |   |   |   | 1 | 2 |
| 5 |   |   |   |   |   | 1 |

Ví dụ ô `dp[0][3]` (`"abca"`): in riêng `a`: `1 + dp[1][3] = 1 + 3 = 4`; ghép với `k = 3`: `dp[1][2] + dp[3][3] = 2 + 1 = 3` → `3`.

Hàm `trace_dp` dưới đây in **từng lựa chọn** khi tính mỗi ô, rồi in bảng.

In [16]:
def trace_dp(s, verbose=True):
    m = len(s)
    dp = [[0] * (m + 1) for _ in range(m + 1)]
    for i in range(m - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, m):
            opts = [(1 + dp[i + 1][j], f"in riêng '{s[i]}': 1 + dp[{i + 1}][{j}]")]
            for k in range(i + 1, j + 1):
                if s[k] == s[i]:
                    inner = dp[i + 1][k - 1] if k - 1 >= i + 1 else 0
                    opts.append((inner + dp[k][j], f"ghép k={k}: dp[{i + 1}][{k - 1}] + dp[{k}][{j}] = {inner}+{dp[k][j]}"))
            dp[i][j] = min(v for v, _ in opts)
            if verbose:
                print(f"dp[{i}][{j}] '{s[i:j + 1]}': " + " | ".join(f"{d} -> {v}" for v, d in opts) + f"  => {dp[i][j]}")
    print("\nBảng dp:")
    print("     " + " ".join(f"{j}{s[j]:<2}" for j in range(m)))
    for i in range(m):
        print(f"{i:>3}: " + " ".join(f"{dp[i][j]:<3}" if j >= i else "   " for j in range(m)))
    return dp[0][m - 1]

r = trace_dp("abcabc")
assert r == 5

dp[4][5] 'bc': in riêng 'b': 1 + dp[5][5] -> 2  => 2
dp[3][4] 'ab': in riêng 'a': 1 + dp[4][4] -> 2  => 2
dp[3][5] 'abc': in riêng 'a': 1 + dp[4][5] -> 3  => 3
dp[2][3] 'ca': in riêng 'c': 1 + dp[3][3] -> 2  => 2
dp[2][4] 'cab': in riêng 'c': 1 + dp[3][4] -> 3  => 3
dp[2][5] 'cabc': in riêng 'c': 1 + dp[3][5] -> 4 | ghép k=5: dp[3][4] + dp[5][5] = 2+1 -> 3  => 3
dp[1][2] 'bc': in riêng 'b': 1 + dp[2][2] -> 2  => 2
dp[1][3] 'bca': in riêng 'b': 1 + dp[2][3] -> 3  => 3
dp[1][4] 'bcab': in riêng 'b': 1 + dp[2][4] -> 4 | ghép k=4: dp[2][3] + dp[4][4] = 2+1 -> 3  => 3
dp[1][5] 'bcabc': in riêng 'b': 1 + dp[2][5] -> 4 | ghép k=4: dp[2][3] + dp[4][5] = 2+2 -> 4  => 4
dp[0][1] 'ab': in riêng 'a': 1 + dp[1][1] -> 2  => 2
dp[0][2] 'abc': in riêng 'a': 1 + dp[1][2] -> 3  => 3
dp[0][3] 'abca': in riêng 'a': 1 + dp[1][3] -> 4 | ghép k=3: dp[1][2] + dp[3][3] = 2+1 -> 3  => 3
dp[0][4] 'abcab': in riêng 'a': 1 + dp[1][4] -> 4 | ghép k=3: dp[1][2] + dp[3][4] = 2+2 -> 4  => 4
dp[0][5] 'abcabc': in riêng

Trace thêm `"aba"` ở chế độ ngắn gọn để thấy rõ nhánh "ghép":

In [17]:
assert trace_dp("aba") == 2

dp[1][2] 'ba': in riêng 'b': 1 + dp[2][2] -> 2  => 2
dp[0][1] 'ab': in riêng 'a': 1 + dp[1][1] -> 2  => 2
dp[0][2] 'aba': in riêng 'a': 1 + dp[1][2] -> 3 | ghép k=2: dp[1][1] + dp[2][2] = 1+1 -> 2  => 2

Bảng dp:
     0a  1b  2a 
  0: 1   2   2  
  1:     1   2  
  2:         1  


## 9. Code nộp LeetCode

Lời giải chính: **Bottom-up + nén chuỗi** (Cách 3), `O(n^3)` thời gian, `O(n^2)` bộ nhớ.

In [18]:
from itertools import groupby

class Solution:
    def strangePrinter(self, s: str) -> int:
        # Nén các ký tự liên tiếp giống nhau: "aaabbb" -> "ab" (không đổi đáp án)
        s = ''.join(ch for ch, _ in groupby(s))
        m = len(s)
        # dp[i][j] = số lượt ít nhất để in s[i..j]; dp[i][i-1] = 0 (đoạn rỗng)
        dp = [[0] * (m + 1) for _ in range(m + 1)]
        for i in range(m - 1, -1, -1):
            dp[i][i] = 1
            for j in range(i + 1, m):
                best = 1 + dp[i + 1][j]                    # in s[i] riêng
                for k in range(i + 1, j + 1):
                    if s[k] == s[i]:                       # lượt in s[i] kéo tới k
                        inner = dp[i + 1][k - 1] if k > i + 1 else 0
                        cand = inner + dp[k][j]
                        if cand < best:
                            best = cand
                dp[i][j] = best
        return dp[0][m - 1]

sol = Solution()
assert sol.strangePrinter("aaabbb") == 2
assert sol.strangePrinter("aba") == 2
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

Ta chứng minh `f(i, j)` (Cách 2/3) bằng đúng số lượt tối thiểu `OPT(i, j)` để in `s[i..j]`.

**(a) `f` đạt được (f ≥ OPT không vi phạm, tức mỗi lựa chọn là một cách in thật).**
- "In riêng": một lượt in `s[i]` tại `i`, cộng một cách in tối ưu của `s[i+1..j]` (không chạm `i`).
- "Ghép với `k`": lấy cách in tối ưu của `s[k..j]`; trong đó có một lượt tạo ra ký tự ở `k`. Có thể giả sử lượt đó là lượt **đầu tiên** của đoạn (in nó trước cũng không mất gì vì nó lộ ở `k` — ta có thể kéo nó ra ngoài cùng). Kéo dài lượt đó sang trái tới `i` (cùng ký tự `s[i] == s[k]`), rồi in `s[i+1..k-1]` bằng một cách tối ưu **sau đó** — đè lên phần kéo dài ở giữa. Tổng số lượt `f(i+1,k-1) + f(k,j)`.

**(b) `OPT ≥ f` (không có cách nào tốt hơn).** Xét một cách in tối ưu, gọi `T` là lượt tạo ra ký tự cuối cùng ở vị trí `i`.
- Nếu `T` không lộ ra ở vị trí nào khác trong `(i, j]`: các lượt khác không chạm `i`... (lượt sau `T` chạm `i` sẽ đè mất; lượt trước `T` tại `i` bị đè và có thể cắt bỏ phần đó). Các lượt còn lại in đúng `s[i+1..j]` → ít nhất `OPT(i+1, j)` lượt, cộng `T`: `≥ 1 + f(i+1, j)`.
- Nếu `T` lộ ra ở vị trí khác, gọi `k` là vị trí **gần `i` nhất**. Mọi lượt sau `T` chạm vào `(i, k)` phải nằm gọn trong `(i, k)` (chạm `i` hay `k` là đè mất `T`), và vì `T` không lộ trong `(i, k)` nên các ô ở đó đều do các lượt sau `T` này tạo ra hoàn toàn → chúng tạo thành một cách in `s[i+1..k-1]`: `≥ f(i+1, k-1)` lượt. Các lượt còn lại (cùng `T`) tạo đúng `s[k..j]`: `≥ f(k, j)` lượt. Tổng `≥ f(i+1,k-1) + f(k,j)`.

Vậy `OPT(i, j) = f(i, j)`. Chi tiết "các lượt trước `T`" được xử lý bằng việc cắt bỏ phần bị đè hoàn toàn (không làm tăng số lượt) — ý tưởng **lập luận trao đổi** quen thuộc. Bộ test ngẫu nhiên ở mục 12 đối chiếu với BFS (không giả định gì) để củng cố niềm tin.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. BFS | `O((σ+1)^n · n^2 σ)` | `O((σ+1)^n)` |
| 2. Memo | `O(n^3)` | `O(n^2)` |
| 3. Bottom-up + nén | `O(m^3)` | `O(m^2)` |
| 4. Chia đôi | `O(n^3)` | `O(n^2)` |

(`σ` = số loại ký tự, `m` = độ dài sau nén.) Với `n = 100`: `n^3 / 6 ≈ 1,7·10^5` bộ `(i, k, j)` — chạy vài chục mili-giây.

In [19]:
random.seed(11)
worst = ''.join(random.choice("ab") for _ in range(100))
worst2 = ''.join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(100))
for name, f in [("memo", strange_printer_memo), ("bottom-up", strange_printer_bottom_up), ("split", strange_printer_split), ("Solution", Solution().strangePrinter)]:
    t = time.perf_counter()
    r1, r2 = f(worst), f(worst2)
    print(f"{name:>10}: {r1}, {r2}  {time.perf_counter() - t:.3f}s")

      memo: 23, 70  0.018s
 bottom-up: 23, 70  0.005s


     split: 23, 70  0.013s
  Solution: 23, 70  0.005s


## 12. Bộ test

Gồm: ví dụ đề, edge case (1 ký tự, toàn cùng ký tự, xen kẽ, đối xứng) và đối chiếu ngẫu nhiên có seed giữa mọi cách với BFS.

In [20]:
def all_methods(s):
    return {
        "memo": strange_printer_memo(s),
        "bottom_up": strange_printer_bottom_up(s),
        "split": strange_printer_split(s),
        "SolutionMemo": SolutionMemo().strangePrinter(s),
        "SolutionBottomUp": SolutionBottomUp().strangePrinter(s),
        "SolutionSplit": SolutionSplit().strangePrinter(s),
        "Solution": Solution().strangePrinter(s),
    }

cases = {
    "aaabbb": 2, "aba": 2, "a": 1, "aaaa": 1, "ab": 2, "abab": 3, "abcba": 3,
    "abcabc": 5, "tbgtgb": 4, "abacaba": 4,
}
for s, want in cases.items():
    for name, got in all_methods(s).items():
        assert got == want, (s, name, got, want)
    if len(s) <= 6:
        assert strange_printer_bfs(s) == want, (s, "bfs")
print("Tất cả", len(cases), "test cố định OK")

Tất cả 10 test cố định OK


In [21]:
random.seed(2024)
for trial in range(150):
    n = random.randint(1, 6)
    s = ''.join(random.choice("abc"[:random.randint(1, 3)]) for _ in range(n))
    want = strange_printer_bfs(s)
    for name, got in all_methods(s).items():
        assert got == want, (s, name, got, want)
for trial in range(300):                      # chuỗi dài hơn: đối chiếu chéo các cách DP
    s = ''.join(random.choice("abcd") for _ in range(random.randint(1, 30)))
    vals = set(all_methods(s).values())
    assert len(vals) == 1, (s, all_methods(s))
print("Test ngẫu nhiên: mọi cách khớp BFS (n<=6) và khớp nhau (n<=30)")

Test ngẫu nhiên: mọi cách khớp BFS (n<=6) và khớp nhau (n<=30)


## 13. Lỗi người mới hay gặp

### Lỗi 1: Đếm số đoạn ký tự liên tiếp

**Code sai:** `return len([ch for ch, _ in groupby(s)])`.

**Vì sao sai:** bỏ qua khả năng in đè — một lượt có thể phủ hai đoạn cùng ký tự ở xa nhau.

In [22]:
def count_runs_wrong(s):
    return len([ch for ch, _ in groupby(s)])

for s in ["aaabbb", "aba", "abcba"]:
    print(s, "sai:", count_runs_wrong(s), "| đúng:", Solution().strangePrinter(s))

aaabbb sai: 2 | đúng: 2
aba sai: 3 | đúng: 2
abcba sai: 5 | đúng: 3


**Code đúng:** DP trên đoạn như Cách 2/3.

### Lỗi 2: Cộng thêm 1 trong nhánh "ghép"

**Code sai:** `res = min(res, 1 + f(i+1, k-1) + f(k, j))`.

**Vì sao sai:** lượt in `s[i]` đã được **gộp** vào lượt in `s[k]` — đã tính trong `f(k, j)`. Cộng 1 là đếm hai lần.

In [23]:
def memo_plus_one_wrong(s):
    @lru_cache(maxsize=None)
    def f(i, j):
        if i > j:
            return 0
        res = 1 + f(i + 1, j)
        for k in range(i + 1, j + 1):
            if s[k] == s[i]:
                res = min(res, 1 + f(i + 1, k - 1) + f(k, j))   # SAI: +1 thừa
        return res
    return f(0, len(s) - 1)

for s in ["aba", "abcba"]:
    print(s, "sai:", memo_plus_one_wrong(s), "| đúng:", Solution().strangePrinter(s))

aba sai: 3 | đúng: 2
abcba sai: 5 | đúng: 3


**Code đúng:** `f(i+1, k-1) + f(k, j)` (không +1).

### Lỗi 3: Thứ tự điền bảng sai trong bottom-up

**Code sai:** `for i in range(m): for j in range(i+1, m): ...` — khi tính `dp[i][j]` cần `dp[i+1][j]` nhưng hàng `i+1` **chưa được tính** (còn 0).

In [24]:
def bottom_up_wrong_order(s):
    m = len(s)
    dp = [[0] * (m + 1) for _ in range(m + 1)]
    for i in range(m):                              # SAI: i phải chạy ngược
        dp[i][i] = 1
        for j in range(i + 1, m):
            best = 1 + dp[i + 1][j]
            for k in range(i + 1, j + 1):
                if s[k] == s[i]:
                    inner = dp[i + 1][k - 1] if k > i + 1 else 0
                    best = min(best, inner + dp[k][j])
            dp[i][j] = best
    return dp[0][m - 1]

for s in ["abcabc", "abab"]:
    print(s, "sai:", bottom_up_wrong_order(s), "| đúng:", Solution().strangePrinter(s))

abcabc sai: 0 | đúng: 5
abab sai: 0 | đúng: 3


**Code đúng:** `i` từ `m-1` về `0` (hoặc điền theo độ dài đoạn tăng dần).

### Lỗi 4: Chỉ ghép với lần xuất hiện **cuối cùng** của `s[i]`

**Code sai:** chỉ thử `k` = vị trí cuối có `s[k] == s[i]` ("kéo xa nhất cho lợi").

**Vì sao sai:** kéo xa nhất không phải lúc nào cũng tốt; đôi khi ghép với lần xuất hiện gần hơn mới tối ưu. Phải thử **mọi** `k`.

In [25]:
def only_last_k_wrong(s):
    @lru_cache(maxsize=None)
    def f(i, j):
        if i > j:
            return 0
        res = 1 + f(i + 1, j)
        ks = [k for k in range(i + 1, j + 1) if s[k] == s[i]]
        if ks:
            k = ks[-1]                                  # SAI: chỉ thử k cuối
            res = min(res, f(i + 1, k - 1) + f(k, j))
        return res
    return f(0, len(s) - 1)

random.seed(1)
for _ in range(2000):
    s = ''.join(random.choice("abc") for _ in range(random.randint(2, 8)))
    if only_last_k_wrong(s) != Solution().strangePrinter(s):
        print("phản ví dụ:", s, "| sai:", only_last_k_wrong(s), "| đúng:", Solution().strangePrinter(s))
        break

phản ví dụ: abaaac | sai: 4 | đúng: 3


**Code đúng:** duyệt mọi `k` có `s[k] == s[i]`.

## 14. Mẹo & nhận dạng pattern

- **"Thao tác trên đoạn liên tiếp, đè lên nhau / xóa rồi ghép lại"** + `n <= 100..500` → nghĩ tới **interval DP** `O(n^3)`.
- Chọn **một phần tử đặc biệt** (đầu đoạn) và hỏi "nó đi chung với ai?" → tách thành hai đoạn con độc lập.
- Ký tự lặp liền nhau thường **nén** được.
- Viết brute force BFS/vét cạn trên trạng thái nhỏ để kiểm chứng công thức DP — rất hữu ích vì công thức interval DP dễ sai tinh tế.
- Họ bài tương tự: Remove Boxes (546), Burst Balloons (312), Zuma Game (488), Minimum Cost to Merge Stones (1000).

## 15. Biến thể

1. **Truy vết các lượt in** — lưu lựa chọn tốt nhất ở mỗi ô, dựng lại danh sách `(l, r, ký tự)` và kiểm tra bằng mô phỏng.
2. **Strange Printer II (LC 1591)** — lưới 2D, hỏi có in được không (topological sort).
3. **Mỗi lượt in có chi phí phụ thuộc ký tự** — thay `1` bằng `cost[c]`.
4. **Giấy ban đầu không trắng** — DP thêm trạng thái "màu nền".

In [26]:
def printer_with_plan(s):
    """Biến thể 1: trả về số lượt và danh sách lượt in (l, r, ch)."""
    n = len(s)

    @lru_cache(maxsize=None)
    def f(i, j):
        if i > j:
            return 0, None
        best = (1 + f(i + 1, j)[0], ('alone',))
        for k in range(i + 1, j + 1):
            if s[k] == s[i]:
                c = f(i + 1, k - 1)[0] + f(k, j)[0]
                if c < best[0]:
                    best = (c, ('join', k))
        return best

    def build(i, j):
        """Trả về list lượt in theo thứ tự thực hiện cho đoạn s[i..j]."""
        if i > j:
            return []
        _, ch = f(i, j)
        if ch[0] == 'alone':
            return [(i, i, s[i])] + build(i + 1, j)
        k = ch[1]
        right = build(k, j)                      # lượt đầu tiên của right phủ từ k
        l0, r0, c0 = right[0]
        assert l0 == k and c0 == s[k]
        right[0] = (i, r0, c0)                   # kéo dài sang trái tới i
        return right[:1] + build(i + 1, k - 1) + right[1:]

    plan = build(0, n - 1)
    return f(0, n - 1)[0], plan

for s in ["abcba", "abcabc", "tbgtgb"]:
    cnt, plan = printer_with_plan(s)
    print(s, cnt, plan)
    assert len(plan) == cnt == Solution().strangePrinter(s)
    assert apply_prints(len(s), plan) == s

abcba 3 [(0, 4, 'a'), (1, 3, 'b'), (2, 2, 'c')]
  in 'a' lên [0,4] -> aaaaa
  in 'b' lên [1,3] -> abbba
  in 'c' lên [2,2] -> abcba
abcabc 5 [(0, 0, 'a'), (1, 1, 'b'), (2, 5, 'c'), (3, 3, 'a'), (4, 4, 'b')]
  in 'a' lên [0,0] -> a_____
  in 'b' lên [1,1] -> ab____
  in 'c' lên [2,5] -> abcccc
  in 'a' lên [3,3] -> abcacc
  in 'b' lên [4,4] -> abcabc
tbgtgb 4 [(0, 0, 't'), (1, 5, 'b'), (2, 4, 'g'), (3, 3, 't')]
  in 't' lên [0,0] -> t_____
  in 'b' lên [1,5] -> tbbbbb
  in 'g' lên [2,4] -> tbgggb
  in 't' lên [3,3] -> tbgtgb


Lưu ý: hàm `build` dựa vào tính chất "lượt đầu tiên của lời giải đoạn `s[k..j]` bắt đầu đúng tại `k` với ký tự `s[k]`" — đúng với cách ta dựng (cả hai nhánh đều đặt lượt chứa vị trí đầu đoạn lên đầu danh sách). `assert` trong code kiểm tra điều đó, và `apply_prints` xác nhận bản in ra đúng chuỗi.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | duyệt mọi trạng thái giấy | hàm mũ | hàm mũ | trọng tài cho chuỗi ≤ 6 |
| 2. Memo | `s[i]` in riêng hoặc ghép với `k` | `O(n^3)` | `O(n^2)` | viết nhanh, ít lỗi chỉ số |
| 3. Bottom-up + nén | như 2, điền bảng | `O(m^3)` | `O(m^2)` | bản nộp chuẩn, không đệ quy |
| 4. Chia đôi | `s[i]==s[j]` → `dp[i][j-1]`, khác → cắt | `O(n^3)` | `O(n^2)` | công thức ngắn, dễ nhớ |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `strangePrinter("abcd")`, `strangePrinter("aabbaa")`, `strangePrinter("abba")`?

**Bài 2 (tính tay).** Lập bảng `dp` (như mục 8) cho `s = "abab"`.

**Bài 3 (điền TODO).** Hoàn thiện `compress(s)` (nén ký tự liên tiếp) **không** dùng `groupby`.

**Bài 4 (tự viết lại).** Viết lại Cách 2 (`my_printer`) mà không nhìn code.

**Bài 5 (mini challenge).** Viết `printer_cost(s, cost)` với `cost` là dict ký tự → chi phí một lượt in ký tự đó; tìm tổng chi phí nhỏ nhất. Kiểm tra: khi mọi chi phí bằng 1 phải trùng `strangePrinter`.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def compress(s):
    # TODO: duyệt s, chỉ thêm ký tự khi khác ký tự cuối đã thêm
    pass

# Bài 4 — TODO
def my_printer(s):
    # TODO: f(i, j) = min(1 + f(i+1, j), f(i+1, k-1) + f(k, j) với s[k] == s[i])
    pass

# Bài 5 — TODO
def printer_cost(s, cost):
    # TODO: như Cách 2 nhưng "in riêng" tốn cost[s[i]] thay vì 1
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: ký tự đôi một khác nhau thì không ghép được.
- Mức 2: `"aabbaa"` nén thành `"aba"`.
- Mức 3: `"abcd"` → 4; `"aabbaa"` → 2; `"abba"` → nén `"aba"` → 2.

**Bài 2.**
- Mức 1: đường chéo toàn 1; `dp[i][i+1] = 2`.
- Mức 2: `dp[0][2]` (`"aba"`) = 2; `dp[1][3]` (`"bab"`) = 2.
- Mức 3: `dp[0][3]`: in riêng `1 + 2 = 3`; ghép `k = 2`: `dp[1][1] + dp[2][3] = 1 + 2 = 3` → 3.

**Bài 3.**
- Mức 1: `out = []`.
- Mức 2: `if not out or out[-1] != ch: out.append(ch)`.
- Mức 3: `return ''.join(out)`.

**Bài 4.**
- Mức 1: `@lru_cache`, `if i > j: return 0`.
- Mức 2: `res = 1 + f(i+1, j)`.
- Mức 3: vòng `k` từ `i+1` tới `j`.

**Bài 5.**
- Mức 1: chi phí của lượt in chứa `s[i]` vẫn chỉ được trả một lần.
- Mức 2: nhánh ghép không cộng chi phí (đã tính trong `f(k, j)`).
- Mức 3: nhánh in riêng: `cost[s[i]] + f(i+1, j)`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().strangePrinter("abcd") == 4
assert Solution().strangePrinter("aabbaa") == 2
assert Solution().strangePrinter("abba") == 2

# Bài 2
assert trace_dp("abab", verbose=False) == 3

# Bài 3
def compress_ans(s):
    out = []
    for ch in s:
        if not out or out[-1] != ch:
            out.append(ch)
    return ''.join(out)

assert compress_ans("aaabbbaac") == "abac" == ''.join(ch for ch, _ in groupby("aaabbbaac"))
print("Bài 1–3 OK")


Bảng dp:
     0a  1b  2a  3b 
  0: 1   2   2   3  
  1:     1   2   2  
  2:         1   2  
  3:             1  
Bài 1–3 OK


In [29]:
# Bài 4
def my_printer_ans(s):
    @lru_cache(maxsize=None)
    def f(i, j):
        if i > j:
            return 0
        res = 1 + f(i + 1, j)
        for k in range(i + 1, j + 1):
            if s[k] == s[i]:
                res = min(res, f(i + 1, k - 1) + f(k, j))
        return res
    return f(0, len(s) - 1)

# Bài 5
def printer_cost_ans(s, cost):
    @lru_cache(maxsize=None)
    def f(i, j):
        if i > j:
            return 0
        res = cost[s[i]] + f(i + 1, j)
        for k in range(i + 1, j + 1):
            if s[k] == s[i]:
                res = min(res, f(i + 1, k - 1) + f(k, j))
        return res
    return f(0, len(s) - 1)

random.seed(4)
for _ in range(200):
    s = ''.join(random.choice("abc") for _ in range(random.randint(1, 12)))
    assert my_printer_ans(s) == Solution().strangePrinter(s)
    assert printer_cost_ans(s, {'a': 1, 'b': 1, 'c': 1}) == Solution().strangePrinter(s)
print(printer_cost_ans("aba", {'a': 5, 'b': 1}), printer_cost_ans("bab", {'a': 5, 'b': 1}))
assert printer_cost_ans("aba", {'a': 5, 'b': 1}) == 6
print("Bài 4–5 OK")

6 6
Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template interval DP "phần tử đầu đi chung với ai":**

```python
s = ''.join(ch for ch, _ in groupby(s))
m = len(s)
dp = [[0] * (m + 1) for _ in range(m + 1)]
for i in range(m - 1, -1, -1):
    dp[i][i] = 1
    for j in range(i + 1, m):
        best = 1 + dp[i + 1][j]
        for k in range(i + 1, j + 1):
            if s[k] == s[i]:
                best = min(best, (dp[i + 1][k - 1] if k > i + 1 else 0) + dp[k][j])
        dp[i][j] = best
return dp[0][m - 1]
```

**Checklist:**

- [ ] Đoạn rỗng trả 0, đoạn 1 ký tự trả 1.
- [ ] Nhánh ghép **không** cộng thêm 1.
- [ ] Thử **mọi** `k` có `s[k] == s[i]`, không chỉ một.
- [ ] Bottom-up: `i` chạy ngược (hoặc theo độ dài tăng dần).
- [ ] Mảng `(m+1) × (m+1)` để không vượt chỉ số `dp[i+1][j]`.
- [ ] Thử `"a"`, `"aaaa"`, `"aba"`, `"abcabc"`.